# Chronos-Bolt-small — representative price baseline

An independent notebook adapted from the workstation's historical **scripts/fm1.py** experiment. This is an additional research baseline; the team's two final submissions are archived in the other two notebooks.

The recovered experiments repeatedly used Chronos-Bolt-small. No original Chronos notebook or complete three-fold comparison of all variants was found, so this notebook is **not labelled the latest or best-performing experiment**. The exact original script is preserved at [archive/chronos/fm1.py](archive/chronos/fm1.py).

**Method:** a pretrained patch-based encoder–decoder forecasts price quantiles. We keep the original 256-observation context, 20-observation horizon, 19 requested quantiles (0.05–0.95), and the fraction of terminal quantiles above the anchor close as a directional score. Bolt supports native levels 0.1–0.9: the historical request's endpoints are clamped to that range and intermediate levels are interpolated by the library. We make the same endpoint clamping explicit to avoid repeated warnings. This fraction is a heuristic, not a calibrated probability. No task-specific training or news embeddings are used. [Official model description](https://huggingface.co/amazon/chronos-bolt-small).

**Packaging changes:** fixed seeds and model revision; explicit input discovery; validation restricted to target years 2020/2021/2022; complete template-based CSV export. Export uses the original project's ±1% direction encoding. It does not reproduce the separate Chronos/news rank-blend candidate. Historical weights, cached scores, and workstation scripts are not required.


## 1. Environment and shared inputs

Use Python 3.11 with PyTorch 2.4.1 (CUDA 12.1 was used for the packaging check). Install PyTorch for your hardware first, then run the optional install cell below. Restart the kernel if an already imported dependency changes.

Restore the repository's shared release data with **node scripts/download_inputs.cjs ./data**. A smaller download for this notebook is:

~~~sh
node scripts/download_inputs.cjs ./data --only kaggle/train.parquet,kaggle/test.parquet,submission_samples.csv
~~~

Run from the repository directory, or attach those files to Kaggle. Set DATA_DIR and TEMPLATE_PATH explicitly if automatic discovery is ambiguous. The pretrained model is downloaded separately from Hugging Face at a pinned revision; it is not part of the competition-data release. For an offline run, set MODEL_PATH to a previously downloaded model directory.


In [ ]:
# Run once in a fresh environment; uncomment if these packages are missing.
# %pip install numpy==1.26.4 pandas==2.2.3 pyarrow==17.0.0 scikit-learn==1.5.2 chronos-forecasting==1.5.3 transformers==4.48.3 accelerate==0.34.2 huggingface-hub==0.27.1 tokenizers==0.21.4 safetensors==0.8.0


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import json
import random
import platform
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from chronos import BaseChronosPipeline

SEED = 42
CONTEXT_LENGTH = 256
HORIZON = 20
QUANTILES = [round(i / 20, 2) for i in range(1, 20)]
BATCH_SIZE = 256
VALIDATION_YEARS = (2020, 2021, 2022)
RUN_VALIDATION = True
RUN_SUBMISSION = True

DATA_DIR = None            # e.g. Path("./data/kaggle")
TEMPLATE_PATH = None       # e.g. Path("./data/submission_samples.csv")
MODEL_PATH = "amazon/chronos-bolt-small"  # or a local model directory
MODEL_REVISION = "772f3d25d38aec6d914c8949dab4462e2d46f5d8"
OUTPUT_DIR = Path("/kaggle/working/chronos") if Path("/kaggle/working").is_dir() else Path("outputs/chronos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE.startswith("cuda") and torch.cuda.is_bf16_supported() else torch.float32
print({"device": DEVICE, "dtype": str(DTYPE), "model": MODEL_PATH})


In [ ]:
def find_input(filename, explicit=None):
    if explicit is not None:
        candidate = Path(explicit)
        if not candidate.is_file():
            raise FileNotFoundError(candidate)
        return candidate.resolve()
    roots = [Path("data"), Path("/kaggle/input")]
    matches = set()
    if DATA_DIR is not None:
        candidate = Path(DATA_DIR) / filename
        if candidate.is_file():
            return candidate.resolve()
    for root in roots:
        if root.is_dir():
            matches.update(p.resolve() for p in root.rglob(filename) if p.is_file())
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one {filename}; found {len(matches)}. Set DATA_DIR or TEMPLATE_PATH explicitly.")
    return next(iter(matches))


def load_prices(filename, last_date=None):
    filters = [("date", "<=", pd.Timestamp(last_date))] if last_date is not None else None
    frame = pd.read_parquet(find_input(filename), columns=["ticker", "date", "close"], filters=filters)
    frame["date"] = pd.to_datetime(frame["date"])
    frame = frame.sort_values(["ticker", "date"]).reset_index(drop=True)
    if frame.duplicated(["ticker", "date"]).any():
        raise ValueError("Duplicate ticker/date keys in price input")
    return frame


def make_contexts(prices, requests):
    """Requests use anchor_date; target-date prices never enter the context."""
    series = {ticker: group.reset_index(drop=True) for ticker, group in prices.groupby("ticker", sort=False)}
    contexts = np.empty((len(requests), CONTEXT_LENGTH), dtype=np.float32)
    for i, row in enumerate(requests.itertuples(index=False)):
        group = series.get(row.ticker)
        if group is None:
            raise ValueError(f"Missing ticker: {row.ticker}")
        dates = group["date"].to_numpy()
        pos = dates.searchsorted(np.datetime64(row.anchor_date))
        if pos >= len(group) or dates[pos] != np.datetime64(row.anchor_date):
            raise ValueError("Missing exact anchor date; no future-date fallback is allowed")
        if pos < CONTEXT_LENGTH:
            raise ValueError("Insufficient context after the historical experiment's warm-up")
        values = group["close"].iloc[pos - CONTEXT_LENGTH + 1:pos + 1].to_numpy(np.float32)
        if not np.isfinite(values[-1]) or values[-1] <= 0:
            raise ValueError("Invalid anchor close")
        contexts[i] = values
    return contexts


def make_validation_requests(prices):
    # The input has already been restricted to the permitted validation period.
    group = prices.groupby("ticker", sort=False)
    requests = prices.rename(columns={"date": "anchor_date", "close": "anchor_close"}).copy()
    requests["target_date"] = group["date"].shift(-HORIZON)
    requests["target_close"] = group["close"].shift(-HORIZON)
    eligible = (
        requests["target_date"].dt.year.isin(VALIDATION_YEARS)
        & (group.cumcount() >= CONTEXT_LENGTH)
        & np.isfinite(requests["anchor_close"]) & (requests["anchor_close"] > 0)
        & np.isfinite(requests["target_close"]) & (requests["target_close"] > 0)
    )
    return requests.loc[eligible].reset_index(drop=True)


def make_submission_requests(template):
    if list(template.columns) != ["ID", "Close"] or not template["ID"].is_unique:
        raise ValueError("Expected a unique-ID submission template with columns ID, Close")
    keys = template["ID"].str.rsplit("_", n=1, expand=True)
    requests = pd.DataFrame({"ticker": keys[0], "target_date": pd.to_datetime(keys[1], errors="raise")})
    # The supplied competition grid uses Monday-Friday business days.
    requests["anchor_date"] = requests["target_date"] - pd.offsets.BDay(HORIZON)
    if (requests["anchor_date"] >= requests["target_date"]).any():
        raise ValueError("Invalid forecast horizon")
    return requests


def predict_direction(contexts, pipeline):
    scores = []
    # Match the library's historical endpoint clamping explicitly.
    effective_quantiles = [min(0.9, max(0.1, q)) for q in QUANTILES]
    with torch.inference_mode():
        for start in range(0, len(contexts), BATCH_SIZE):
            batch = contexts[start:start + BATCH_SIZE]
            quantiles, _ = pipeline.predict_quantiles(
                context=torch.from_numpy(batch), prediction_length=HORIZON,
                quantile_levels=effective_quantiles,
            )
            terminal = quantiles[:, -1, :].float().cpu().numpy()
            if terminal.shape != (len(batch), len(QUANTILES)) or not np.isfinite(terminal).all():
                raise ValueError("Invalid model quantile output")
            scores.append((terminal > batch[:, -1, None]).mean(axis=1))
    return np.concatenate(scores) if scores else np.empty(0, dtype=float)


def encode_submission(template, contexts, scores):
    if len(template) != len(contexts) or len(template) != len(scores):
        raise ValueError("Submission prediction count does not match the template")
    direction = np.where(scores >= 0.5, 1.0, -1.0)
    result = template.copy()
    # Direction encoding, not a calibrated prediction of return magnitude.
    result["Close"] = np.round(contexts[:, -1].astype(float) * (1 + 0.01 * direction), 2)
    if not np.isfinite(result["Close"]).all() or not (result["Close"] > 0).all():
        raise ValueError("Invalid predicted close")
    assert result["ID"].equals(template["ID"])
    return result


## 2. Load the fixed pretrained model

Chronos-Bolt-small is used in zero-shot mode. No fitting, scaling, tuning, or model selection uses the held-out competition year. Each input window ends at its anchor date. The validation routine below reads only the permitted training-period rows; submission inference never reads target labels.


In [ ]:
model_kwargs = {} if Path(MODEL_PATH).is_dir() else {"revision": MODEL_REVISION}
pipeline = BaseChronosPipeline.from_pretrained(
    MODEL_PATH, device_map=DEVICE, torch_dtype=DTYPE, **model_kwargs
)
pipeline.model.eval()
environment = {
    "python": platform.python_version(),
    "packages": {name: version(name) for name in ["torch", "numpy", "pandas", "pyarrow", "scikit-learn", "chronos-forecasting", "transformers", "accelerate", "huggingface-hub", "tokenizers", "safetensors"]},
    "cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0) if DEVICE.startswith("cuda") else "cpu",
    "dtype": str(DTYPE), "seed": SEED, "model": MODEL_PATH,
    "model_revision": MODEL_REVISION if not Path(MODEL_PATH).is_dir() else "user-supplied local snapshot",
    "context_length": CONTEXT_LENGTH, "horizon": HORIZON, "quantiles": QUANTILES,
}
(OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")


## 3. Validation on three historical target years

Report 2020, 2021 and 2022 separately, with the same fixed model and 0.5 threshold. Zero-shot inference has no expanding-window fitting step. Targets are used only for scoring after inference; price contexts include observations through the anchor only. Flat returns are excluded from directional metrics. The first 256 rows per stock form the historical warm-up. No year-wide ranking or top-K selection is used.

These are diagnostic results for this repackaged baseline, not evidence that it was the best historical Chronos variant or that its pretraining corpus was audited.


In [ ]:
if RUN_VALIDATION:
    validation_prices = load_prices("train.parquet", last_date="2022-12-31")
    validation = make_validation_requests(validation_prices)
    validation_contexts = make_contexts(validation_prices, validation)
    validation["score_up"] = predict_direction(validation_contexts, pipeline)
    reports = []
    for year in VALIDATION_YEARS:
        part = validation.loc[validation["target_date"].dt.year.eq(year)].copy()
        part = part.loc[part["target_close"].ne(part["anchor_close"])]
        if part.empty:
            raise ValueError(f"No eligible validation samples for {year}")
        actual_up = part["target_close"].gt(part["anchor_close"]).to_numpy()
        score = part["score_up"].to_numpy()
        reports.append({
            "target_year": year, "n": len(part),
            "directional_hit": float(np.mean((score >= 0.5) == actual_up)),
            "always_up_hit": float(actual_up.mean()),
            "auc_up": float(roc_auc_score(actual_up, score)) if len(np.unique(actual_up)) == 2 else None,
        })
    validation.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
    (OUTPUT_DIR / "validation_metrics.json").write_text(json.dumps(reports, indent=2) + "\n")
    print(pd.DataFrame(reports).to_string(index=False))
    del validation_contexts


## 4. Generate a submission CSV

The template fixes the row order and required IDs. Forecasting reads price history only through each ID's anchor, 20 business days before the target. The fraction-of-quantiles direction is encoded as anchor_close × (1 ± 0.01), following the project's historical directional CSV convention. This export adapter is new; it is not one of the two final submitted solutions. No held-out performance is calculated or displayed.

Output: **outputs/chronos/submission.csv** locally, or **/kaggle/working/chronos/submission.csv** on Kaggle. The output folder also contains environment information and the optional historical validation report.


In [ ]:
if RUN_SUBMISSION:
    template = pd.read_csv(find_input("submission_samples.csv", explicit=TEMPLATE_PATH))
    submission_requests = make_submission_requests(template)
    inference_prices = load_prices("test.parquet", last_date=submission_requests["anchor_date"].max())
    submission_contexts = make_contexts(inference_prices, submission_requests)
    submission_scores = predict_direction(submission_contexts, pipeline)
    submission = encode_submission(template, submission_contexts, submission_scores)
    submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
    print(f"Saved {len(submission):,} rows to {OUTPUT_DIR / 'submission.csv'}")
    del submission_contexts
